# Toxic Gaming Train — User-Reply Networks (Per Subreddit)

This notebook constructs **directed user-reply networks** from toxicity-scored Reddit datasets across eight gaming subreddits.  
Each network models how users interact within their community by linking **comment authors (sources)** to the **users they reply to (targets)**.

### Objectives
- Transform JSONL comment data into **directed reply networks** (C→C and C→S edges).  
- Compute **dataset-level summaries** (shape, missing data, authors, time span, toxicity).  
- Generate **network metrics** (nodes, edges, density, reciprocity, degree stats).  
- Export **Gephi-ready CSV files** for visualization and comparative analysis.

### Edge Types
- **C→C** — Comment → Parent comment’s author (`t1_*`)  
- **C→S** — Comment → Submission author (`t3_*`)  

Each subreddit produces its own network, data summary, and exported CSVs to support downstream visualization and analysis of toxic interaction patterns.


## Column dictionary (what each field means)

| column | meaning |
|---|---|
| `subreddit` | Subreddit name (e.g., `albiononline`). |
| `post_id` | Root submission ID (base36). |
| `title` | Root submission title. |
| `comment_id` | Comment ID (base36) for this row. |
| `parent_id` | Thing ID of parent: `t1_<comment_id>` (comment) or `t3_<post_id>` (submission). |
| `author` | Author of **this** comment. |
| `parent_author` | Author of parent comment or submission (target of the reply). |
| `parent_kind` | Parent type prefix (`t1` for comment, `t3` for post). |
| `parent_key` | Base36 ID of the parent without prefix (if provided). |
| `record_type` | Row type (expect `comment` here). |
| `body_raw` | Original raw comment text. |
| `body` | Cleaned/lowercased comment text (if your pipeline produced it). |
| `created_utc` | Unix timestamp (seconds since 1970-01-01). |
| `created_dt` | Converted datetime (derived from `created_utc`). |
| `score` | Reddit vote score for the comment. |
| `toxicity_flag` | Boolean; whether comment is flagged as toxic. |
| `toxicity_categories` | Dict of category booleans (hate, harassment, etc.). |
| `toxicity_scores` | Dict of category floats per category. |
| `__key` | Internal key like `t1_<comment_id>`. |


### Setup: Imports and Configuration

This cell imports all required Python libraries and defines key configuration parameters for the project:

In [32]:
import os, json, math
from collections import Counter
from typing import Dict, Any
import pandas as pd
import numpy as np
import networkx as nx

files = [
    {"name": "WoW",                  "path": "../data_clean_scored/scored_WoW.SAMPLE.jsonl"},
    {"name": "ElderScrollsOnline",   "path": "../data_clean_scored/scored_ElderScrollsOnline.SAMPLE.jsonl"},
    {"name": "SkyChildrenoftheLight","path": "../data_clean_scored/scored_SkyChildrenoftheLight.SAMPLE.jsonl"},
    {"name": "AlbionOnline",         "path": "../data_clean_scored/scored_AlbionOnline.SAMPLE.jsonl"},
    {"name": "DarkSouls",            "path": "../data_clean_scored/scored_DarkSouls.SAMPLE.jsonl"},
    {"name": "LiesofP",              "path": "../data_clean_scored/scored_LiesofP.SAMPLE.jsonl"},
    {"name": "HollowKnight",         "path": "../data_clean_scored/scored_HollowKnight.SAMPLE.jsonl"},
    {"name": "Ninesols",             "path": "../data_clean_scored/scored_Ninesols.SAMPLE.jsonl"},
]
OUT_DIR = "../user_reply_networks"
EXCLUDE_USERS = {"[deleted]", "AutoModerator", None, ""}

### Load Subreddit JSONL data into DataFrames

In [34]:
def load_jsonl(path) -> pd.DataFrame:
    rows, bad = [], 0
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s: 
                continue
            try:
                rows.append(json.loads(s))
            except json.JSONDecodeError:
                bad += 1
    df = pd.DataFrame(rows)
    
    # Derives columns: created_dt (human-readable datetime of comment creation), 
    # parent_kind (whether parent is a comment (t1) or submission (t3)), and parent_key (the base ID of the parent comment/post)
    if "created_dt" not in df and "created_utc" in df:
        df["created_dt"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")
    if "parent_kind" not in df.columns and "parent_id" in df.columns:
        df["parent_kind"] = df["parent_id"].astype(str).str.slice(0,2)
    if "parent_key" not in df.columns and "parent_id" in df.columns:
        df["parent_key"] = df["parent_id"].astype(str).str.split("_").str[-1]
    return df

### Basic Dataset Overview

This function summarizes each subreddit’s dataset to give a quick snapshot of its structure and quality. It reports metrics like dataset size, missing values, unique authors, overall toxicity rate, time span, and the distribution of comment-to-comment (C→C) vs. comment-to-submission (C→S) interactions.

In [36]:
def compute_basic_overview(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    # Shape
    n_rows, n_cols = df.shape

    # Record type counts if present (post vs comment)
    if "record_type" in df.columns:
        rtc = df["record_type"].astype(str).str.lower().value_counts().to_dict()
    else:
        rtc = {}

    # Check for missing data (selected important cols only)
    cols_to_check = ["author","parent_author","root_author","comment_id","parent_id","post_id",
                     "created_utc","created_dt","toxicity_flag","parent_kind","parent_key"]
    missing = {c: int(df[c].isna().sum()) for c in cols_to_check if c in df.columns}

    # Time span
    if "created_dt" in df.columns and df["created_dt"].notna().any():
        tmin = pd.to_datetime(df["created_dt"]).min()
        tmax = pd.to_datetime(df["created_dt"]).max()
        days = (tmax - tmin).days if pd.notna(tmin) and pd.notna(tmax) else np.nan
    else:
        tmin = tmax = pd.NaT
        days = np.nan

    # Number of unique authors
    uniq_authors = df["author"].nunique(dropna=True) if "author" in df.columns else np.nan

    # Toxicity (overall)
    if "toxicity_flag" in df.columns:
        tox_rate = float(pd.to_numeric(df["toxicity_flag"], errors="coerce").mean())
    else:
        tox_rate = np.nan

    # Extracts comment-only subset for interaction building
    if "record_type" in df.columns:
        comments = df.loc[df["record_type"].astype(str).str.lower().eq("comment")].copy()
    else:
        comments = df.copy()
    comments = comments if not comments.empty else pd.DataFrame(columns=df.columns)

    # C→C vs C→S counts (based on parent_kind)
    if "parent_kind" in comments.columns:
        c2c = int((comments["parent_kind"] == "t1").sum())
        c2s = int((comments["parent_kind"] == "t3").sum())
    else:
        c2c = c2s = np.nan

    # Summary dict of above computed values
    row = {
        "subreddit": name,
        "rows": n_rows,
        "cols": n_cols,
        "unique_authors": int(uniq_authors) if pd.notna(uniq_authors) else np.nan,
        "toxicity_rate": tox_rate,
        "tmin": tmin,
        "tmax": tmax,
        "timespan_days": days,
        "comments_rows": int(len(comments)),
        "c2c_rows": c2c,
        "c2s_rows": c2s,
        "record_type_counts": rtc,
        "missing_selected": missing,
    }
    # Print a compact overview
    print(f"\n--- {name} | Data Overview ---")
    print(f"shape={n_rows}x{n_cols} | unique_authors={row['unique_authors']} | "
          f"tox_rate={None if pd.isna(tox_rate) else round(tox_rate,3)}")
    if pd.notna(tmin) and pd.notna(tmax):
        print(f"time span: {tmin} → {tmax} ({days} days)")
    if rtc:
        print(f"record_type counts: {rtc}")
    print(f"comments={row['comments_rows']} (C→C={c2c}, C→S={c2s})")
    print(f"missing (selected): {missing}")
    return row

### Building the User-Reply Edgelist

This function constructs a directed edgelist showing who replies to whom within a subreddit. It links each commenter (**Source**) to the author they replied to (**Target**), handling both comment-to-comment (C→C) and comment-to-submission (C→S) cases. The result is a weighted table of interactions, counting the number of replies and calculating the fraction of toxic edges between each user pair.

In [38]:
def make_edgelist(comments: pd.DataFrame) -> pd.DataFrame:
    # Map for parent comment author (t1 fallback)
    comment_author_map = (
        comments.loc[:, ["comment_id","author"]]
        .dropna()
        .set_index("comment_id")["author"]
        if "comment_id" in comments.columns and "author" in comments.columns else pd.Series(dtype=object)
    )

    # Start with parent_author if already present
    tgt = comments.get("parent_author")
    if tgt is None: 
        tgt = pd.Series(index=comments.index, dtype=object)
    tgt = tgt.copy()

    # t3 fallback: use root_author
    mask_t3_na = comments.get("parent_kind", pd.Series(index=comments.index)).eq("t3") & tgt.isna()
    if "root_author" in comments.columns:
        tgt.loc[mask_t3_na] = comments.loc[mask_t3_na, "root_author"]

    # t1 fallback: map parent_key -> author
    mask_t1_na = comments.get("parent_kind", pd.Series(index=comments.index)).eq("t1") & tgt.isna()
    if "parent_key" in comments.columns and not comment_author_map.empty:
        tgt.loc[mask_t1_na] = comments.loc[mask_t1_na, "parent_key"].map(comment_author_map)

    el = pd.DataFrame({
        "Source": comments.get("author"),
        "Target": tgt,
        "parent_kind": comments.get("parent_kind"),
        "created_dt": comments.get("created_dt"),
        "toxicity_flag": comments.get("toxicity_flag"),
    }).dropna(subset=["Source","Target"])

    el = el[~el["Source"].isin(EXCLUDE_USERS) & ~el["Target"].isin(EXCLUDE_USERS)]
    el = el[el["Source"] != el["Target"]]  # optional: remove self-loops

    # Weight = number of replies from Source to Target (edge weight)
    # Toxic_Edge_Frac = average of toxicity_flag on those replies (i.e., fraction of the Source→Target messages that were flagged toxic)
    weighted = (el.groupby(["Source","Target"], as_index=False)
                  .agg(Weight=("toxicity_flag","count"),
                       Toxic_Edge_Frac=("toxicity_flag", lambda x: float(np.mean(pd.to_numeric(x, errors="coerce"))) if len(x)>0 else np.nan)))
    return weighted

### Creating Node Attributes

Generates a table of user-level statistics for each subreddit network. Each node (user) includes their total number of comments and average toxicity rate, forming the attribute data used in Gephi visualizations.

In [40]:
def make_nodes(df: pd.DataFrame) -> pd.DataFrame:
    comments = df.loc[df["record_type"].astype(str).str.lower().eq("comment")] if "record_type" in df.columns else df
    comments = comments[~comments["author"].isin(EXCLUDE_USERS)]
    node_stats = (comments.groupby("author") 
                    .agg(comments=("author","size"), 
                         toxicity_rate=("toxicity_flag", lambda x: float(np.mean(pd.to_numeric(x, errors="coerce")))))
                    .reset_index()
                    .rename(columns={"author":"Id"}))
    return node_stats

### Summarizing Network Structure

Analyzes each subreddit’s user-reply network to produce key structural metrics. It computes node and edge counts, density, reciprocity, degree statistics, and highlights the largest connected component and top interacting users, providing an overview of the community’s interaction patterns.

In [42]:
def summarize_network(edges: pd.DataFrame, nodes: pd.DataFrame, name: str) -> Dict[str, Any]:
    # Directed graph
    G = nx.DiGraph()
    G.add_nodes_from(nodes["Id"].tolist())
    G.add_weighted_edges_from(edges[["Source","Target","Weight"]].itertuples(index=False, name=None))

    # Compute basic size and density
    n = G.number_of_nodes()
    m = G.number_of_edges()
    density = nx.density(G)
    
    # Compute weakly connected components and reciprocity
    if n > 0 and m > 0:
        wccs = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
        giant = len(wccs[0]) if wccs else 0
        giant_share = giant / n if n else 0.0
        reciprocity = nx.reciprocity(G)
    else:
        giant = 0; giant_share = 0.0; reciprocity = np.nan

    # In/out degree stats
    indeg = dict(G.in_degree())
    outdeg = dict(G.out_degree()) 
    avg_in  = float(np.mean(list(indeg.values()))) if n else 0
    med_in  = float(np.median(list(indeg.values()))) if n else 0
    avg_out = float(np.mean(list(outdeg.values()))) if n else 0
    med_out = float(np.median(list(outdeg.values()))) if n else 0

    # Identify top users
    top_out = sorted(outdeg.items(), key=lambda x: x[1], reverse=True)[:5]
    top_in  = sorted(indeg.items(),  key=lambda x: x[1], reverse=True)[:5]

    # Merge toxicity info
    tox_map = nodes.set_index("Id")["toxicity_rate"].to_dict()
    def _fmt(lst):
        return "; ".join([f"{u}({k}; tox={None if math.isnan(tox_map.get(u, np.nan)) else round(tox_map.get(u, np.nan),3)})"
                          for u,k in lst]) if lst else "(none)"

    print(f"\n--- {name} — Network Summary ---")
    print(f"Nodes={n} | Edges={m} | Density={density:.6f}")
    print(f"Largest WCC: {giant} ({giant_share:.1%}) | Reciprocity: {np.nan if pd.isna(reciprocity) else round(reciprocity,4)}")
    print(f"Avg/Med In: {avg_in:.2f}/{med_in:.2f} | Avg/Med Out: {avg_out:.2f}/{med_out:.2f}")
    print(f"Top Out: {_fmt(top_out)}")
    print(f"Top In : {_fmt(top_in)}")

    return {
        "net_nodes": n,
        "net_edges": m,
        "density": density,
        "largest_wcc_nodes": giant,
        "largest_wcc_share": giant_share,
        "reciprocity": reciprocity,
        "avg_in_degree": avg_in,
        "med_in_degree": med_in,
        "avg_out_degree": avg_out,
        "med_out_degree": med_out,
    }

### Exporting Gephi Files

Saves each subreddit’s processed data into Gephi-compatible CSV files. The `edges_<subreddit>.csv` file stores user-to-user interactions, while `nodes_<subreddit>.csv` contains user attributes for visualization and analysis in Gephi.

In [44]:
def ensure_dir(p): os.makedirs(p, exist_ok=True)

def export_for_gephi(edges: pd.DataFrame, nodes: pd.DataFrame, name: str, out_dir: str):
    ensure_dir(out_dir)  
    edges_out = edges[["Source","Target","Weight"]].copy()
    edges_out.to_csv(os.path.join(out_dir, f"edges_{name}.csv"), index=False)
    nodes_out = nodes.copy()
    nodes_out.to_csv(os.path.join(out_dir, f"nodes_{name}.csv"), index=False) 

### Building and Exporting All Subreddit Networks

Runs the complete pipeline across all eight subreddit datasets.  

For each file, it loads the data, prints a basic overview, builds the user-reply edgelist and node attributes, summarizes the resulting network, and exports both Gephi files and a combined `summary_overview.csv` containing all subreddit statistics.

In [46]:
def build_all(files, out_dir=OUT_DIR):
    ensure_dir(out_dir)
    overview_rows = []

    for item in files:
        name, path = item["name"], item["path"]
        print(f"\n[LOAD] {name}: {path}")
        df = load_jsonl(path)
        if df.empty:
            print("  -> Empty file; skipping.")
            continue

        # Basic overview (prints + row for CSV)
        basic_row = compute_basic_overview(df, name)

        # Interation subset
        comments = df.loc[df["record_type"].astype(str).str.lower().eq("comment")] if "record_type" in df.columns else df
        edges = make_edgelist(comments)
        nodes = make_nodes(df)
 
        # Attach in/out degree to nodes for Gephi
        Gtmp = nx.DiGraph()
        Gtmp.add_nodes_from(nodes["Id"].tolist())
        Gtmp.add_weighted_edges_from(edges[["Source","Target","Weight"]].itertuples(index=False, name=None))
        indeg  = dict(Gtmp.in_degree())
        outdeg = dict(Gtmp.out_degree())
        nodes["in_degree"]  = nodes["Id"].map(indeg).fillna(0).astype(int)
        nodes["out_degree"] = nodes["Id"].map(outdeg).fillna(0).astype(int)

        # Network Summary (prints + dict for CSV)
        net_row = summarize_network(edges, nodes, name)

        # Export
        export_for_gephi(edges, nodes, name, out_dir=out_dir)

        # Merge both summary dicts
        row = {**basic_row, **net_row}
        overview_rows.append(row)

    # Write one combined overview CSV
    if overview_rows:
        pd.DataFrame(overview_rows).to_csv(os.path.join(out_dir, "summary_overview.csv"), index=False)
        print(f"\n[OK] Wrote Gephi CSVs + overview to: {os.path.abspath(out_dir)}")

# Execute
build_all(files, OUT_DIR)


[LOAD] WoW: ../data_clean_scored/scored_WoW.SAMPLE.jsonl

--- WoW | Data Overview ---
shape=11933x28 | unique_authors=6331 | tox_rate=0.042
time span: 2025-08-25 07:37:39 → 2025-10-15 20:22:38 (51 days)
record_type counts: {'comment': 11601, 'post': 332}
comments=11601 (C→C=6815, C→S=4786)
missing (selected): {'author': 0, 'parent_author': 20, 'root_author': 0, 'comment_id': 332, 'parent_id': 0, 'post_id': 0, 'created_utc': 0, 'created_dt': 0, 'toxicity_flag': 11, 'parent_kind': 0, 'parent_key': 0}

--- WoW — Network Summary ---
Nodes=6316 | Edges=10444 | Density=0.000262
Largest WCC: 6218 (98.4%) | Reciprocity: 0.2679
Avg/Med In: 1.65/0.00 | Avg/Med Out: 1.65/1.00
Top Out: olvekstoneheid_2006(54; tox=0.0); exhvoid(45; tox=0.019); abyssalanarkay(31; tox=0.098); ugaeismyamongusname(29; tox=0.03); automoderator(23; tox=0.0)
Top In : yourresidentferal(558; tox=0.0); early_conflict_160(260; tox=0.2); turtvaiz(259; tox=0.167); ex0ll(169; tox=0.333); olvekstoneheid_2006(149; tox=0.0)

[LOAD